# Exploring a microscopy image stack with Xarray

Xarray is useful beyond geoscience: microscopy images are also naturally multi-dimensional. This short example uses the `cells3d` tutorial dataset, which contains two fluorescence channels and a segmentation mask. It demonstrates how named dimensions and coordinates make it easier to inspect, select, crop, and visualize an image volume.

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

## Load and inspect the data

The tutorial loader supplies a small example dataset, so no separate download or domain-specific reader is needed. Inspecting the dimensions first helps make the array structure explicit: `c` is the channel, `z` is depth, and `y` and `x` are the image axes.

In [ ]:
ds = xr.tutorial.load_dataset("cells3d")
ds

The `images` variable holds the fluorescence data and `mask` holds segmentation labels. The channel coordinate lets us select a named channel instead of remembering an integer axis position. These coordinates identify positions in the sample image; they do not encode physical distances.

In [ ]:
images = ds["images"]
print("Image dimensions:", images.dims)
print("Image shape:", images.shape)
print("Channels:", images.coords["c"].values)
print("Mask dimensions:", ds["mask"].dims)

## Select a channel and depth plane

Use `.sel` for coordinate labels such as the channel name. Use `.isel` when selecting by integer position. Here, `method="nearest"` makes the depth selection work even if the requested coordinate is not an exact match.

In [ ]:
nuclei = images.sel(c="nuclei")
plane = nuclei.sel(z=9, method="nearest")
plane.plot(cmap="gray")

A labeled crop can be made along the image axes while keeping the depth stack. Taking the maximum across `z` produces a maximum-intensity projection: each pixel shows the brightest value encountered through the selected depth range.

In [ ]:
crop = nuclei.isel(z=slice(10, 20), y=slice(80, 180), x=slice(80, 180))
projection = crop.max(dim="z")
projection.plot(cmap="gray")

## Summarize intensity across the image

Reducing over `y` and `x` leaves one value per depth plane. Xarray keeps the `z` coordinate attached, so the result can be plotted directly and remains easy to label or align with other depth-dependent measurements.

In [ ]:
depth_profile = nuclei.mean(dim=("y", "x"))
depth_profile.plot()
plt.title("Mean nuclei-channel intensity by depth")
plt.show()

## Compare an image with its segmentation mask

The mask labels regions in the same image coordinates. Overlaying the nonzero labels on the fluorescence plane illustrates how Xarray aligns arrays using their named coordinates. The mask is a segmentation result; it is not another fluorescence channel.

In [ ]:
labels = ds["mask"].sel(z=plane.coords["z"])
fig, ax = plt.subplots(figsize=(7, 6))
plane.plot(ax=ax, cmap="gray", add_colorbar=False)
labels.where(labels > 0).plot(ax=ax, cmap="tab20", alpha=0.45, add_colorbar=False)
ax.set_title("Nuclei fluorescence with segmentation labels")
plt.show()

## Takeaways

- Named dimensions make the structure of an image volume easier to understand.
- Coordinate labels support readable selection and keep plots aligned after reductions.
- The same Xarray operations used for geospatial grids also work for microscopy data.

This example builds on the non-geoscience tutorial request in [issue #172](https://github.com/xarray-contrib/xarray-tutorial/issues/172).